# Notebook 05 — Exportação das Bases Finais

**Projeto:** Mineração de Dados em Saúde — PNS 2019
**Estudo:** Artrite e Reumatismo em Idosos Brasileiros
**Pesquisador:** Pedro Dias Soares
**Orientador:** Prof. Dr. Luis Enrique Zárate — PUC Minas (LICAP)

---

## Objetivo

Empacotar as bases **já trabalhadas** (pré-processadas no NB03/03b e discretizadas no NB04)
no formato de entrega solicitado pelo orientador. Para **cada um dos dois desenhos amostrais**,
este notebook gera **um banco de dados SQLite (`.db`)** e **um arquivo `.csv`**.

Este notebook **não transforma** os dados — apenas **consolida e exporta** os datasets finais
para distribuição. A modelagem de ML fica no próximo notebook.

| Desenho | Entrada (saída do NB04) | Saída (NB05) |
|---|---|---|
| 1 — Artrite pura | `discretizacao/dataset_discretizado.csv` | `bases_finais/desenho1_artrite_pura.{db, csv}` |
| 2 — Artrite + comorbidades | `discretizacao_comorbidades/dataset_discretizado.csv` | `bases_finais/desenho2_comorbidades.{db, csv}` |

Cada `.db` contém duas tabelas: **`dados`** (o dataset trabalhado) e **`metadados`**
(proveniência: origem, dimensões, distribuição do alvo, data de geração).

## 1 · Configuração

`CONFIG_DESENHOS` centraliza, para cada desenho, **de onde ler** (o `dataset_discretizado.csv`
do NB04) e **como nomear** os arquivos de entrega. Todas as bases vão para uma única pasta
`data/results/bases_finais/`, facilitando o envio ao orientador.

In [1]:
import os, json, sqlite3, warnings
from datetime import datetime
import pandas as pd

warnings.filterwarnings('ignore')

# Pasta única onde ficam as bases finais (entrega)
DIR_SAIDA = '../data/results/bases_finais/'
os.makedirs(DIR_SAIDA, exist_ok=True)

# Cada desenho: de onde ler (saída do NB04) e como nomear a base final
CONFIG_DESENHOS = {
    'puro': {
        'rotulo':  'Desenho 1 — Artrite pura',
        'entrada': '../data/results/discretizacao/dataset_discretizado.csv',
        'base':    'desenho1_artrite_pura',   # nome dos arquivos .db e .csv
        'tabela':  'dados',                    # tabela de dados dentro do .db
    },
    'comorbidades': {
        'rotulo':  'Desenho 2 — Artrite + comorbidades',
        'entrada': '../data/results/discretizacao_comorbidades/dataset_discretizado.csv',
        'base':    'desenho2_comorbidades',
        'tabela':  'dados',
    },
}

RANDOM_STATE = 42
print(f'✅ Ambiente configurado. Bases finais → {DIR_SAIDA}')
for k, c in CONFIG_DESENHOS.items():
    print(f'   • {c["rotulo"]}: {c["entrada"]}')

✅ Ambiente configurado. Bases finais → ../data/results/bases_finais/
   • Desenho 1 — Artrite pura: ../data/results/discretizacao/dataset_discretizado.csv
   • Desenho 2 — Artrite + comorbidades: ../data/results/discretizacao_comorbidades/dataset_discretizado.csv


## 2 · Função de exportação

`exportar_base` lê o dataset discretizado de um desenho, faz uma checagem de sanidade
(coluna-alvo `Label` presente, ausência de nulos — o dataset deve sair limpo do NB04) e grava
os dois formatos de entrega. O `.db` é regravado do zero a cada execução (idempotente).

In [2]:
def exportar_base(chave, cfg, dir_saida=DIR_SAIDA):
    """Lê o dataset discretizado (NB04) de um desenho e grava a base final
    em SQLite (.db) e CSV (.csv). Retorna um dicionário-resumo."""
    # 1) Carregar o dataset já trabalhado
    assert os.path.exists(cfg['entrada']), (
        f"Não encontrei {cfg['entrada']}.\n"
        f"Rode o NB04 (discretização) para o desenho {chave!r} antes deste notebook."
    )
    df = pd.read_csv(cfg['entrada'])

    # 2) Sanidade: alvo presente e sem nulos
    assert 'Label' in df.columns, "Coluna-alvo 'Label' ausente no dataset."
    n_nulos = int(df.isna().sum().sum())
    if n_nulos:
        print(f'  ⚠️ {n_nulos} valores nulos encontrados — verifique o NB03/NB04.')

    dist = {str(int(k)): int(v)
            for k, v in df['Label'].value_counts().sort_index().items()}

    caminho_csv = os.path.join(dir_saida, cfg['base'] + '.csv')
    caminho_db  = os.path.join(dir_saida, cfg['base'] + '.db')

    # 3) CSV final (cópia de entrega)
    df.to_csv(caminho_csv, index=False, encoding='utf-8-sig')

    # 4) Banco SQLite: tabela de dados + tabela de metadados
    if os.path.exists(caminho_db):
        os.remove(caminho_db)              # regrava do zero
    con = sqlite3.connect(caminho_db)
    try:
        df.to_sql(cfg['tabela'], con, index=False, if_exists='replace')
        meta = pd.DataFrame([
            ('projeto',      'Artrite e Reumatismo em Idosos — PNS 2019'),
            ('desenho',      cfg['rotulo']),
            ('origem',       cfg['entrada']),
            ('notebook',     '05_exportacao_bases'),
            ('tabela_dados', cfg['tabela']),
            ('n_linhas',     str(len(df))),
            ('n_colunas',    str(df.shape[1])),
            ('alvo',         'Label (0=Saudável, 1=Com Artrite)'),
            ('distribuicao', json.dumps(dist)),
            ('gerado_em',    datetime.now().strftime('%Y-%m-%d %H:%M:%S')),
        ], columns=['chave', 'valor'])
        meta.to_sql('metadados', con, index=False, if_exists='replace')
    finally:
        con.close()

    print(f'  ✅ {cfg["rotulo"]}: {df.shape[0]:,} × {df.shape[1]} | alvo={dist}')
    print(f'     • CSV → {caminho_csv}')
    print(f'     • DB  → {caminho_db}  (tabelas: {cfg["tabela"]}, metadados)')
    return {'desenho': chave, 'linhas': df.shape[0], 'colunas': df.shape[1],
            'distribuicao': dist, 'csv': caminho_csv, 'db': caminho_db}

## 3 · Execução para os dois desenhos

Percorre `CONFIG_DESENHOS` e gera as quatro entregas (2 `.db` + 2 `.csv`).

In [3]:
resumos = []
for chave, cfg in CONFIG_DESENHOS.items():
    print(f'▶ Exportando {cfg["rotulo"]} ...')
    resumos.append(exportar_base(chave, cfg))
    print()

pd.DataFrame(resumos)[['desenho', 'linhas', 'colunas', 'distribuicao', 'db', 'csv']]

▶ Exportando Desenho 1 — Artrite pura ...


  ✅ Desenho 1 — Artrite pura: 4,826 × 57 | alvo={'0': 4332, '1': 494}
     • CSV → ../data/results/bases_finais/desenho1_artrite_pura.csv
     • DB  → ../data/results/bases_finais/desenho1_artrite_pura.db  (tabelas: dados, metadados)

▶ Exportando Desenho 2 — Artrite + comorbidades ...


  ✅ Desenho 2 — Artrite + comorbidades: 8,357 × 55 | alvo={'0': 4332, '1': 4025}
     • CSV → ../data/results/bases_finais/desenho2_comorbidades.csv
     • DB  → ../data/results/bases_finais/desenho2_comorbidades.db  (tabelas: dados, metadados)



,desenho,linhas,colunas,distribuicao,db,csv
0,puro,4826,57,"{'0': 4332, '1': 494}",../data/results/bases_finais/desenho1_artrite_...,../data/results/bases_finais/desenho1_artrite_...
1,comorbidades,8357,55,"{'0': 4332, '1': 4025}",../data/results/bases_finais/desenho2_comorbid...,../data/results/bases_finais/desenho2_comorbid...


## 4 · Verificação (round-trip)

Relê cada base **a partir do SQLite** para confirmar que o `.db` reproduz fielmente o dataset
(mesmas dimensões) e exibe os metadados gravados e as primeiras linhas.

In [4]:
for r in resumos:
    con = sqlite3.connect(r['db'])
    try:
        df_db   = pd.read_sql('SELECT * FROM dados', con)
        df_meta = pd.read_sql('SELECT * FROM metadados', con)
    finally:
        con.close()

    print(f'🔎 {os.path.basename(r["db"])}: relido {df_db.shape[0]:,} × {df_db.shape[1]}')
    assert df_db.shape == (r['linhas'], r['colunas']), 'Divergência no round-trip do SQLite!'
    display(df_meta)
    display(df_db.head())
    print()

print('✅ Round-trip OK — as bases SQLite reproduzem os CSVs trabalhados.')

🔎 desenho1_artrite_pura.db: relido 4,826 × 57


,chave,valor
0,projeto,Artrite e Reumatismo em Idosos — PNS 2019
1,desenho,Desenho 1 — Artrite pura
2,origem,../data/results/discretizacao/dataset_discreti...
3,notebook,05_exportacao_bases
4,tabela_dados,dados
5,n_linhas,4826
6,n_colunas,57
7,alvo,"Label (0=Saudável, 1=Com Artrite)"
8,distribuicao,"{""0"": 4332, ""1"": 494}"
9,gerado_em,2026-06-10 11:44:33


,VDF004,P006,P013,P02801,P03202,VDD004A,N001,N010,N011,C006_Mulher,...,P00614_Sim,P00615_Sim,P00616_Sim,IMC_cat,C008_cat,P035_cat,J012_cat,P029_cat,Padrao_Alimentar_cat,Label
0,3.0,3.000000,3.0,0.0,0.0,4.0,1.0,3.0,0.0,1,...,0,1,0,2,0,1,1,0,3,1
1,5.0,5.488458,4.0,1.0,0.0,1.0,0.0,0.0,0.0,0,...,1,0,0,3,0,0,0,1,2,1
2,4.0,7.000000,2.0,0.0,0.0,1.0,3.0,3.0,1.0,0,...,0,0,0,2,0,3,1,0,0,1
3,4.0,7.000000,3.0,0.0,0.0,1.0,1.0,3.0,2.0,1,...,0,0,1,1,0,3,0,0,3,1
4,3.0,7.000000,1.0,1.0,0.0,0.0,2.0,0.0,0.0,0,...,0,0,0,0,0,0,1,1,3,1


🔎 desenho2_comorbidades.db: relido 8,357 × 55


,chave,valor
0,projeto,Artrite e Reumatismo em Idosos — PNS 2019
1,desenho,Desenho 2 — Artrite + comorbidades
2,origem,../data/results/discretizacao_comorbidades/dat...
3,notebook,05_exportacao_bases
4,tabela_dados,dados
5,n_linhas,8357
6,n_colunas,55
7,alvo,"Label (0=Saudável, 1=Com Artrite)"
8,distribuicao,"{""0"": 4332, ""1"": 4025}"
9,gerado_em,2026-06-10 11:44:33


,VDF004,P006,P013,P02801,P03202,VDD004A,N001,N010,N011,C006_Mulher,...,P00614_Sim,P00615_Sim,P00616_Sim,IMC_cat,C008_cat,P035_cat,J012_cat,P029_cat,Padrao_Alimentar_cat,Label
0,3.0,2.0,1.0,0.0,0.0,1.0,4.0,2.0,2.0,1,...,0,0,0,2,0,0,1,0,1,1
1,4.0,3.0,4.0,0.0,0.0,1.0,2.0,1.0,0.0,1,...,0,0,0,1,1,3,3,0,0,1
2,7.0,7.0,2.0,0.0,0.0,4.0,2.0,0.0,0.0,0,...,0,1,0,2,1,0,2,0,2,1
3,6.0,2.0,2.0,0.0,0.0,6.0,1.0,0.0,1.0,1,...,0,0,0,3,1,2,0,0,0,1
4,4.0,7.0,1.0,0.0,0.0,1.0,2.0,1.0,1.0,1,...,0,0,0,2,0,2,2,0,0,1



✅ Round-trip OK — as bases SQLite reproduzem os CSVs trabalhados.


## 5 · Síntese

Artefatos gerados em `data/results/bases_finais/`:

```
bases_finais/
├── desenho1_artrite_pura.db      ← SQLite (tabelas: dados, metadados)
├── desenho1_artrite_pura.csv     ← Desenho 1 — artrite pura
├── desenho2_comorbidades.db      ← SQLite (tabelas: dados, metadados)
└── desenho2_comorbidades.csv     ← Desenho 2 — artrite + comorbidades
```

Cada base é o dataset **100% trabalhado** (pré-processado + discretizado), pronto para a
modelagem de Machine Learning na etapa seguinte.